# T-10 — Load Nodes into Neo4j

Load all node labels in T-10's strict scope into the AuraDB instance using the official `neo4j` Python driver. Every load is **idempotent** (uses `MERGE`), so this notebook is safe to re-run and safe to run on top of the T-09 sample data already in the database.

## Scope

Five node labels — exactly the ones named in T-10:

| Label | Source | Expected count |
|---|---|---|
| `:Genre` | `data/raw/ml-100k/u.genre` | 19 |
| `:User` | `data/raw/ml-100k/u.user` | 943 |
| `:Movie` | `data/raw/ml-100k/u.item` + `data/enriched/enriched_movies.csv` | 1,682 |
| `:Director` | `data/enriched/enriched_movies.csv` (`directors` column) | ~1,300–1,500 unique |
| `:Actor` | `data/enriched/enriched_movies.csv` (`cast` column) | ~10,000 unique |

## Not in scope

- Relationships (`RATED`, `IN_GENRE`, `DIRECTED_BY`, `FEATURES_ACTOR`) — separate tasks.
- `:Occupation` and `:Gender` nodes — deferred to the task that adds their `HAS_*` edges.
- Any TMDB API calls — enrichment is already done in `02_tmdb_enrichment.ipynb`.

## How the code is split

- **`src/etl.py`** — file readers only (pandas, regex, JSON-in-CSV parsing). Boring plumbing.
- **This notebook** — the **inline Cypher** that does the actual loading. This is the KG-specific learning: `UNWIND` + `MERGE` + parameter binding.
- **`src/graph.py`** — driver connection + `run_query` / `run_write_query` helpers.

## Setup

In [2]:
# Make `from src...` work when running from notebooks/
import sys
from pathlib import Path

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.etl import (
    read_genres, read_users, read_movies, read_directors, read_actors,
    to_records,
)
from src.graph import run_query, run_write_query, close_driver

## Constraints + fulltext index

Re-creating the schema artifacts from `queries/validate_schema_sample.cyp` (T-09) so this notebook is self-sufficient. `IF NOT EXISTS` makes the statements no-ops when they're already present.

In [3]:
schema_statements = [
    "CREATE CONSTRAINT user_id     IF NOT EXISTS FOR (u:User)     REQUIRE u.userId   IS UNIQUE",
    "CREATE CONSTRAINT movie_id    IF NOT EXISTS FOR (m:Movie)    REQUIRE m.movieId  IS UNIQUE",
    "CREATE CONSTRAINT genre_name  IF NOT EXISTS FOR (g:Genre)    REQUIRE g.name     IS UNIQUE",
    "CREATE CONSTRAINT director_nm IF NOT EXISTS FOR (d:Director) REQUIRE d.name     IS UNIQUE",
    "CREATE CONSTRAINT actor_name  IF NOT EXISTS FOR (a:Actor)    REQUIRE a.name     IS UNIQUE",
    "CREATE FULLTEXT INDEX movie_title_search IF NOT EXISTS FOR (m:Movie) ON EACH [m.title]",
]
for stmt in schema_statements:
    run_write_query(stmt)
print(f"Created/verified {len(schema_statements)} schema artifacts.")

Created/verified 6 schema artifacts.


## Optional teardown

Uncomment **only** if you want to wipe every node and relationship in the database before loading. Destructive — only do this on a throwaway AuraDB instance.

In [4]:
# run_write_query("MATCH (n) DETACH DELETE n")
# print("Database wiped.")

## 1 — Genres

19 names from `u.genre`. The simplest loader: a list of strings, one MERGE per name via `UNWIND`.

In [5]:
genre_names = read_genres()
print(f"Read {len(genre_names)} genres from u.genre:")
print("  " + ", ".join(genre_names))

cypher = """
UNWIND $names AS name
MERGE (:Genre {name: name})
"""
run_write_query(cypher, {"names": genre_names})

count = run_query("MATCH (:Genre) RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count} :Genre nodes.")

Read 19 genres from u.genre:
  unknown, Action, Adventure, Animation, Children's, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western
Neo4j now has 19 :Genre nodes.


## 2 — Users

943 users from `u.user`. T-10 strict scope keeps only `userId` and `age` on the node; gender / occupation / zip stay in the DataFrame but are dropped before the load.

In [6]:
users_df = read_users()
print(f"Read {len(users_df)} users from u.user.")
users_df.head()

Read 943 users from u.user.


,userId,age
0,1,24
1,2,53
2,3,23
3,4,24
4,5,33


In [7]:
cypher = """
UNWIND $rows AS row
MERGE (u:User {userId: row.userId})
SET u.age = row.age
"""
run_write_query(cypher, {"rows": to_records(users_df)})

count = run_query("MATCH (:User) RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count} :User nodes.")

Neo4j now has 943 :User nodes.


## 3 — Movies

1,682 movies. Reads `u.item` (movieId, title, year) and left-joins `enriched_movies.csv` (tmdbId). Movies that TMDB couldn't match still create a Movie node — just without the `tmdbId` property. The `FOREACH … CASE` trick is the canonical Cypher idiom for **conditional property assignment** (set the property only when the value is not null).

In [8]:
movies_df = read_movies()
unmatched = movies_df["tmdbId"].isna().sum()
print(f"Read {len(movies_df)} movies.")
print(f"  matched to TMDB:   {len(movies_df) - unmatched}")
print(f"  unmatched (no tmdbId): {unmatched}")
movies_df.head()

Read 1682 movies.
  matched to TMDB:   1679
  unmatched (no tmdbId): 3


,movieId,title,year,tmdbId
0,1,Toy Story,1995.0,862
1,2,GoldenEye,1995.0,710
2,3,Four Rooms,1995.0,5
3,4,Get Shorty,1995.0,8012
4,5,Copycat,1995.0,1710


In [9]:
cypher = """
UNWIND $rows AS row
MERGE (m:Movie {movieId: row.movieId})
SET m.title = row.title, m.year = row.year
FOREACH (_ IN CASE WHEN row.tmdbId IS NULL THEN [] ELSE [1] END |
  SET m.tmdbId = row.tmdbId
)
"""
run_write_query(cypher, {"rows": to_records(movies_df)})

count = run_query("MATCH (:Movie) RETURN count(*) AS n")[0]["n"]
with_tmdb = run_query("MATCH (m:Movie) WHERE m.tmdbId IS NOT NULL RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count} :Movie nodes ({with_tmdb} with a tmdbId).")

Neo4j now has 1682 :Movie nodes (1679 with a tmdbId).


## 4 — Directors

One movie can have multiple directors (the Coen brothers, the Wachowskis, anthology films like *Four Rooms*). The reader flattens every `directors` cell across the enriched CSV and dedupes by name.

In [10]:
director_names = read_directors()
print(f"Read {len(director_names)} unique directors from enriched_movies.csv.")
print("  Sample:", director_names[:5])

cypher = """
UNWIND $names AS name
MERGE (:Director {name: name})
"""
run_write_query(cypher, {"names": director_names})

count = run_query("MATCH (:Director) RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count} :Director nodes.")

Read 1128 unique directors from enriched_movies.csv.
  Sample: ['Aaron Kim Johnston', 'Aaron Speiser', 'Abel Ferrara', 'Adam Resnick', 'Adil El Arbi']
Neo4j now has 1128 :Director nodes.


## 5 — Actors

Top-10 cast per matched movie, flattened and deduped. Many actors appear in multiple movies — the dedup is doing real work here. Edges with billing order land in T-11 (`FEATURES_ACTOR`).

In [11]:
actor_names = read_actors()
print(f"Read {len(actor_names)} unique actors from enriched_movies.csv.")
print("  Sample:", actor_names[:5])

cypher = """
UNWIND $names AS name
MERGE (:Actor {name: name})
"""
run_write_query(cypher, {"names": actor_names})

count = run_query("MATCH (:Actor) RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count} :Actor nodes.")

Read 9301 unique actors from enriched_movies.csv.
  Sample: ['A. Bailey Chapin', 'A. J. Langer', 'A.C. Smith', 'A.J. Johnson', 'Aaron Dozier']
Neo4j now has 9301 :Actor nodes.


## Summary — node counts by label

Final tally across every label in the database. If you ran on a fresh instance (or after the optional teardown), expect roughly 5 labels and ~13,000+ total nodes. If you ran on top of T-09 sample data, expect the same labels plus `:Occupation` and `:Gender` left behind from T-09.

In [12]:
result = run_query("""
MATCH (n)
RETURN labels(n)[0] AS label, count(*) AS n
ORDER BY n DESC
""")
print(f"{'label':<12} {'count':>8}")
print("-" * 22)
for row in result:
    print(f"{row['label']:<12} {row['n']:>8}")

label           count
----------------------
Actor            9301
Movie            1682
Director         1128
User              943
Genre              19


## Idempotency check (re-run after the cells above)

Run the summary cell again. Counts should be **identical** — no double-loads. That's the MERGE pattern doing its job.

In [13]:
close_driver()
print("Driver closed.")

Driver closed.
